# FleetLogix
## Proyecto Integrador Módulo 2 — Data Science

**Avance 1: Generación y validación de datos**

**Base de datos:** PostgreSQL  
**Herramientas:** Python, Pandas, NumPy, Faker y DBeaver

### Objetivo
Construir y validar una base de datos relacional para una empresa de logística, simulando vehículos, conductores, rutas, viajes, entregas y mantenimientos.

**Volumen generado:** 505.650 registros distribuidos en 6 tablas.

---

## 1. Configuración del entorno de trabajo

En esta sección se importan las bibliotecas necesarias para la generación, manipulación y validación de datos.

Se utilizan Python y sus bibliotecas para trabajar con los registros, mientras que PostgreSQL permite almacenar y consultar la información de FleetLogix.

In [1]:
import sys
import pandas as pd
import numpy as np
import faker
import psycopg2
import tabulate

print("¡FleetLogix está listo!")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

¡FleetLogix está listo!
Python: 3.14.8
Pandas: 3.1.0
NumPy: 2.5.4


## 2. Conexión a la base de datos PostgreSQL

Se establece una conexión entre Python y la base de datos relacional `fleetlogix`, alojada en PostgreSQL.

La conexión utiliza la biblioteca `psycopg2` y solicita la contraseña mediante `getpass`, evitando mostrarla directamente en pantalla.

**Parámetros de conexión:**
- Servidor: `localhost`
- Puerto: `5432`
- Base de datos: `fleetlogix`
- Usuario: `fleetlogix_user`

Esta conexión permite ejecutar consultas SQL y realizar controles de calidad sobre los datos generados.

In [9]:
import psycopg2
from getpass import getpass

# Pedimos la contraseña sin mostrarla en pantalla
password = getpass("Contraseña de FleetLogix: ")

try:
    conexion = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="fleetlogix",
        user="fleetlogix_user",
        password=password,
        connect_timeout=10
    )

    print("✅ ¡Python se conectó correctamente a FleetLogix!")

    conexion.close()
    print("✅ Conexión cerrada correctamente.")

except psycopg2.Error as error:
    print("❌ No se pudo conectar a PostgreSQL.")
    print("Tipo de error:", type(error).__name__)

Contraseña de FleetLogix:  ········


✅ ¡Python se conectó correctamente a FleetLogix!
✅ Conexión cerrada correctamente.


## 3. Generación de datos sintéticos

### 3.1. Generación de vehículos

Se generan 200 registros sintéticos de vehículos para simular la flota de una empresa de logística.

Cada vehículo cuenta con información sobre su identificación, tipo, capacidad de carga, combustible y estado operativo.

**Objetivo:** disponer de una flota simulada que permita analizar la capacidad de transporte, la asignación de viajes y las operaciones de mantenimiento.

La generación utiliza una semilla aleatoria para favorecer la reproducibilidad de los resultados.

In [7]:
import pandas as pd
import random
from datetime import date, timedelta

# Semilla para obtener resultados reproducibles
random.seed(42)

# Cantidad de vehículos solicitada
cantidad_vehiculos = 200

# Características de nuestra flota
tipos_vehiculo = {
    "motorcycle": 30,
    "van": 850,
    "pickup": 1200,
    "truck": 5000
}

vehiculos = []

for i in range(1, cantidad_vehiculos + 1):
    tipo = random.choice(list(tipos_vehiculo.keys()))

    vehiculo = {
        "license_plate": f"FLX{i:04d}",
        "vehicle_type": tipo,
        "capacity_kg": tipos_vehiculo[tipo],
        "fuel_type": random.choice({
    "motorcycle": ["gasoline", "electric"],
    "van": ["diesel", "gasoline", "electric"],
    "pickup": ["diesel", "gasoline"],
    "truck": ["diesel", "gasoline"]}[tipo]),
"acquisition_date": date(2026, 10, 26)
            - timedelta(days=random.randint(30, 2500)),
        "status": random.choices(
            ["active", "maintenance", "inactive"],
            weights=[80, 15, 5]
        )[0]
    }

    vehiculos.append(vehiculo)

# Convertimos los registros en un DataFrame
vehicles_df = pd.DataFrame(vehiculos)

# Controles de calidad iniciales
assert len(vehicles_df) == 200
assert vehicles_df["license_plate"].is_unique
assert vehicles_df.notna().all().all()
assert (vehicles_df["capacity_kg"] > 0).all()

print("✅ Vehículos generados:", len(vehicles_df))
print("✅ Patentes únicas y datos básicos validados")

display(vehicles_df.head(10))

✅ Vehículos generados: 200
✅ Patentes únicas y datos básicos validados


,license_plate,vehicle_type,capacity_kg,fuel_type,acquisition_date,status
0,FLX0001,motorcycle,30,gasoline,2023-08-27,active
1,FLX0002,van,850,electric,2025-08-03,active
2,FLX0003,motorcycle,30,electric,2026-05-19,active
3,FLX0004,van,850,diesel,2021-01-26,active
4,FLX0005,van,850,electric,2020-08-16,active
5,FLX0006,truck,5000,gasoline,2026-08-31,active
6,FLX0007,van,850,electric,2021-12-30,active
7,FLX0008,van,850,diesel,2022-12-18,active
8,FLX0009,truck,5000,diesel,2022-09-17,maintenance
9,FLX0010,pickup,1200,diesel,2021-08-02,active


In [8]:
# Revisar combinaciones poco realistas
motos_diesel = vehicles_df[
    (vehicles_df["vehicle_type"] == "motorcycle") &
    (vehicles_df["fuel_type"] == "diesel")
]

print("Motocicletas diésel encontradas:", len(motos_diesel))

display(motos_diesel.head())

Motocicletas diésel encontradas: 0


,license_plate,vehicle_type,capacity_kg,fuel_type,acquisition_date,status


### 3.2. Generación de conductores

Se generan 400 conductores sintéticos para simular el personal de FleetLogix.

Cada conductor cuenta con un código único, nombre, apellido, número de licencia, fecha de vencimiento, teléfono, fecha de contratación y estado laboral.

**Objetivo:** disponer de información confiable para asignar conductores a los viajes y controlar la vigencia de sus licencias.

Se aplican controles de calidad para verificar la unicidad de los códigos y licencias, la ausencia de valores nulos y la consistencia de las fechas.

In [9]:
from faker import Faker

fake = Faker("es_AR")
Faker.seed(42)

print("Faker está listo para generar conductores.")

Faker está listo para generar conductores.


In [10]:
import pandas as pd
import random
from datetime import date, timedelta
from faker import Faker

fake = Faker("es_AR")
Faker.seed(42)
random.seed(42)

conductores = []

for i in range(1, 401):
    conductor = {
        "employee_code": f"EMP{i:04d}",
        "first_name": fake.first_name()[:100],
        "last_name": fake.last_name()[:100],
        "license_number": f"LIC{i:06d}",
        "license_expiry": date(2026, 11, 3)
            + timedelta(days=random.randint(30, 1825)),
        "phone": f"11{random.randint(10000000, 99999999)}",
        "hire_date": date(2026, 11, 3)
            - timedelta(days=random.randint(30, 2500)),
        "status": random.choices(
            ["active", "inactive"],
            weights=[85, 15]
        )[0]
    }

    conductores.append(conductor)

drivers_df = pd.DataFrame(conductores)

print("Conductores generados:", len(drivers_df))
display(drivers_df.head(10))

Conductores generados: 400


,employee_code,first_name,last_name,license_number,license_expiry,phone,hire_date,status
0,EMP0001,Luca,Gonzalez,LIC000001,2030-07-04,1124942603,2026-06-24,active
1,EMP0002,Victoria,Diaz,LIC000002,2028-04-17,1139958838,2025-03-12,active
2,EMP0003,Abril,Peralta,LIC000003,2030-09-18,1183197857,2025-10-13,active
3,EMP0004,Uma,Rodriguez,LIC000004,2027-02-06,1113999315,2025-09-16,active
4,EMP0005,Ciro,Gonzalez,LIC000005,2029-10-02,1190801586,2026-06-18,active
5,EMP0006,Delfina,Gimenez,LIC000006,2030-12-08,1197226012,2020-08-24,active
6,EMP0007,Martina,Diaz,LIC000007,2029-06-09,1189089901,2023-08-22,active
7,EMP0008,Lara,Molina,LIC000008,2026-12-16,1131429110,2022-01-07,active
8,EMP0009,Delfina,Nuñez,LIC000009,2027-10-17,1138898923,2022-12-26,active
9,EMP0010,Alma Valentina,Gonzalez,LIC000010,2029-01-19,1122981052,2022-09-25,active


In [11]:
# Controles de calidad de los conductores

assert len(drivers_df) == 400
assert drivers_df["employee_code"].is_unique
assert drivers_df["license_number"].is_unique
assert drivers_df.notna().all().all()
assert (drivers_df["license_expiry"] > drivers_df["hire_date"]).all()
assert drivers_df["status"].isin(["active", "inactive"]).all()

print("✅ Los 400 conductores pasaron los controles de calidad.")
print("✅ Códigos y licencias únicos.")
print("✅ Sin valores nulos.")
print("✅ Fechas y estados validados.")

✅ Los 400 conductores pasaron los controles de calidad.
✅ Códigos y licencias únicos.
✅ Sin valores nulos.
✅ Fechas y estados validados.


### 3.3. Generación de rutas

Se generan 50 rutas sintéticas para representar los recorridos de la empresa FleetLogix.

Cada ruta contiene información sobre la ciudad de origen, la ciudad de destino, la distancia recorrida, la duración estimada y el costo de peajes.

**Objetivo:** disponer de rutas que permitan planificar viajes y analizar distancias, tiempos y costos operativos.

In [12]:

import pandas as pd
import random

# Ciudades de origen y destino
ciudades = [
    "Buenos Aires", "Córdoba", "Rosario",
    "Mendoza", "Salta", "Tucumán",
    "Santiago del Estero", "Santa Fe",
    "La Plata", "Mar del Plata"
]

random.seed(43)
rutas = []
combinaciones = [
    (origen, destino)
    for origen in ciudades
    for destino in ciudades
    if origen != destino
]

for i, (origen, destino) in enumerate(
    random.sample(combinaciones, 50), start=1
):
    distancia = round(random.uniform(80, 1500), 2)
    duracion = round(distancia / random.uniform(55, 85), 2)
    peaje = round(random.uniform(0, 15000), 2)

    rutas.append({
        "route_code": f"RUT{i:04d}",
        "origin_city": origen,
        "destination_city": destino,
        "distance_km": distancia,
        "estimated_duration_hours": duracion,
        "toll_cost": peaje
    })

routes_df = pd.DataFrame(rutas)

assert len(routes_df) == 50
assert routes_df["route_code"].is_unique
assert routes_df.notna().all().all()
assert (routes_df["distance_km"] > 0).all()
assert (routes_df["estimated_duration_hours"] > 0).all()
assert (routes_df["toll_cost"] >= 0).all()

print("✅ Rutas generadas:", len(routes_df))
print("✅ Controles básicos de calidad superados")
display(routes_df.head(10))


✅ Rutas generadas: 50
✅ Controles básicos de calidad superados


,route_code,origin_city,destination_city,distance_km,estimated_duration_hours,toll_cost
0,RUT0001,Buenos Aires,Tucumán,1008.66,13.54,915.59
1,RUT0002,Salta,Buenos Aires,1279.61,18.62,9707.17
2,RUT0003,Rosario,Buenos Aires,1107.22,17.06,9961.65
3,RUT0004,Santiago del Estero,Tucumán,703.08,11.96,1202.27
4,RUT0005,Tucumán,Rosario,533.75,6.57,4021.91
5,RUT0006,Córdoba,Salta,1114.17,14.82,1055.27
6,RUT0007,Santiago del Estero,Salta,1490.55,19.91,4455.09
7,RUT0008,La Plata,Salta,1357.10,17.66,5423.58
8,RUT0009,Santa Fe,Buenos Aires,696.27,10.06,6223.86
9,RUT0010,La Plata,Tucumán,1178.59,17.93,9674.48


In [13]:

import pandas as pd
import random
from datetime import datetime, timedelta

# Primero generamos una muestra de 10 viajes
random.seed(44)

viajes = []

for i in range(1, 11):
    salida = datetime(2026, 1, 1) + timedelta(
        days=random.randint(0, 300),
        hours=random.randint(0, 23)
    )

    duracion = random.randint(2, 15)
    llegada = salida + timedelta(hours=duracion)

    viaje = {
        "vehicle_id": random.randint(1, 200),
        "driver_id": random.randint(1, 400),
        "route_id": random.randint(1, 50),
        "departure_datetime": salida,
        "arrival_datetime": llegada,
        "fuel_consumed_liters": round(random.uniform(10, 150), 2),
        "total_weight_kg": round(random.uniform(100, 1000), 2),
        "status": "completed"
    }

    viajes.append(viaje)

trips_test_df = pd.DataFrame(viajes)

print("✅ Viajes de prueba generados:", len(trips_test_df))
display(trips_test_df)


✅ Viajes de prueba generados: 10


,vehicle_id,driver_id,route_id,departure_datetime,arrival_datetime,fuel_consumed_liters,total_weight_kg,status
0,180,60,12,2026-07-29 16:00:00,2026-07-30 02:00:00,63.14,360.61,completed
1,3,52,11,2026-04-26 03:00:00,2026-04-26 14:00:00,118.42,719.94,completed
2,97,303,26,2026-06-05 22:00:00,2026-06-06 10:00:00,117.25,703.03,completed
3,73,317,11,2026-07-03 02:00:00,2026-07-03 09:00:00,25.68,163.06,completed
4,28,321,19,2026-02-18 08:00:00,2026-02-18 15:00:00,14.90,447.29,completed
5,49,67,17,2026-09-02 15:00:00,2026-09-02 20:00:00,114.60,798.90,completed
6,175,224,46,2026-01-23 21:00:00,2026-01-24 12:00:00,15.30,742.22,completed
7,121,323,30,2026-06-18 11:00:00,2026-06-19 02:00:00,92.60,586.48,completed
8,189,351,23,2026-04-30 15:00:00,2026-05-01 00:00:00,96.28,976.01,completed
9,3,115,1,2026-01-23 08:00:00,2026-01-23 12:00:00,91.21,643.45,completed


In [14]:

# Controles iniciales de los viajes

assert len(trips_test_df) == 10
assert trips_test_df.notna().all().all()

assert (
    trips_test_df["arrival_datetime"] >
    trips_test_df["departure_datetime"]
).all()

assert trips_test_df["vehicle_id"].between(1, 200).all()
assert trips_test_df["driver_id"].between(1, 400).all()
assert trips_test_df["route_id"].between(1, 50).all()

assert (trips_test_df["fuel_consumed_liters"] > 0).all()
assert (trips_test_df["total_weight_kg"] > 0).all()

print("✅ Los 10 viajes pasaron los controles iniciales.")
print("✅ Fechas, identificadores y valores básicos correctos.")


✅ Los 10 viajes pasaron los controles iniciales.
✅ Fechas, identificadores y valores básicos correctos.


In [15]:

import random
import pandas as pd
from datetime import datetime, timedelta

random.seed(44)

cantidad_viajes = 100000
viajes = []

# Vehículos y sus capacidades
capacidades = vehicles_df["capacity_kg"].tolist()

for i in range(cantidad_viajes):

    vehicle_id = random.randint(1, len(vehicles_df))
    driver_id = random.randint(1, len(drivers_df))
    route_id = random.randint(1, len(routes_df))

    salida = datetime(2026, 1, 1) + timedelta(
        days=random.randint(0, 300),
        hours=random.randint(0, 23)
    )

    duracion = random.randint(2, 15)
    llegada = salida + timedelta(hours=duracion)

    capacidad = capacidades[vehicle_id - 1]
    peso = round(random.uniform(1, capacidad), 2)

    viaje = {
        "vehicle_id": vehicle_id,
        "driver_id": driver_id,
        "route_id": route_id,
        "departure_datetime": salida,
        "arrival_datetime": llegada,
        "fuel_consumed_liters": round(
            random.uniform(10, 150), 2
        ),
        "total_weight_kg": peso,
        "status": "completed"
    }

    viajes.append(viaje)

trips_df = pd.DataFrame(viajes)

print("✅ Viajes generados:", len(trips_df))
display(trips_df.head(10))


✅ Viajes generados: 100000


,vehicle_id,driver_id,route_id,departure_datetime,arrival_datetime,fuel_consumed_liters,total_weight_kg,status
0,105,267,35,2026-03-01 05:00:00,2026-03-01 13:00:00,14.07,7.53,completed
1,30,292,1,2026-02-21 05:00:00,2026-02-21 19:00:00,148.53,2569.05,completed
2,178,325,25,2026-07-27 22:00:00,2026-07-28 10:00:00,20.30,392.02,completed
3,73,317,11,2026-02-27 16:00:00,2026-02-27 19:00:00,47.80,706.75,completed
4,81,56,41,2026-05-29 01:00:00,2026-05-29 08:00:00,76.82,1930.01,completed
5,49,97,9,2026-05-13 23:00:00,2026-05-14 04:00:00,16.08,932.10,completed
6,175,224,46,2026-01-20 23:00:00,2026-01-21 12:00:00,128.22,833.97,completed
7,85,190,31,2026-08-21 18:00:00,2026-08-22 08:00:00,42.66,459.91,completed
8,117,378,44,2026-06-27 19:00:00,2026-06-28 10:00:00,113.27,1168.04,completed
9,72,81,1,2026-04-25 00:00:00,2026-04-25 11:00:00,100.55,640.68,completed


### 3.4. Generación de viajes

Se generan 100.000 viajes sintéticos para simular las operaciones de transporte de FleetLogix.

Cada viaje se relaciona con un vehículo, un conductor y una ruta, e incluye fechas de salida y llegada, consumo de combustible, peso transportado y estado del viaje.

**Objetivo:** disponer de información para analizar las operaciones logísticas, los tiempos de traslado, el consumo de combustible y la utilización de vehículos.

Se realizan controles de calidad para verificar la cantidad de registros, la ausencia de valores nulos y la consistencia de las fechas.

In [16]:

# Validación de los 100.000 viajes

assert len(trips_df) == 100000
assert trips_df.notna().all().all()

# Validar fechas
assert (
    trips_df["arrival_datetime"] >
    trips_df["departure_datetime"]
).all()

# Validar identificadores
assert trips_df["vehicle_id"].between(1, 200).all()
assert trips_df["driver_id"].between(1, 400).all()
assert trips_df["route_id"].between(1, 50).all()

# Validar valores positivos
assert (trips_df["fuel_consumed_liters"] > 0).all()
assert (trips_df["total_weight_kg"] > 0).all()

# Validar capacidad de cada vehículo
capacidades = vehicles_df["capacity_kg"].to_numpy()
limites = capacidades[trips_df["vehicle_id"].to_numpy() - 1]

assert (trips_df["total_weight_kg"].to_numpy() <= limites).all()

print("✅ Los 100.000 viajes pasaron los controles de calidad.")
print("✅ Fechas, identificadores, combustible y pesos validados.")
print("✅ Ningún viaje supera la capacidad de su vehículo.")


✅ Los 100.000 viajes pasaron los controles de calidad.
✅ Fechas, identificadores, combustible y pesos validados.
✅ Ningún viaje supera la capacidad de su vehículo.


In [17]:

# Diagnóstico de coherencia de los viajes

print("Cantidad de viajes:", len(trips_df))

print("\nEstados de los viajes:")
print(trips_df["status"].value_counts())

print("\nDuración de los viajes (horas):")
duraciones = (
    trips_df["arrival_datetime"]
    - trips_df["departure_datetime"]
).dt.total_seconds() / 3600

print(duraciones.describe())

print("\nConsumo de combustible (litros):")
print(trips_df["fuel_consumed_liters"].describe())

print("\nPeso transportado (kg):")
print(trips_df["total_weight_kg"].describe())


Cantidad de viajes: 100000

Estados de los viajes:
status
completed    100000
Name: count, dtype: int64

Duración de los viajes (horas):
count    100000.000000
mean          8.488980
std           4.023573
min           2.000000
25%           5.000000
50%           8.000000
75%          12.000000
max          15.000000
dtype: float64

Consumo de combustible (litros):
count    100000.000000
mean         79.875499
std          40.494210
min          10.010000
25%          44.750000
50%          79.830000
75%         114.990000
max         150.000000
Name: fuel_consumed_liters, dtype: float64

Peso transportado (kg):
count    100000.000000
mean        818.775466
std        1135.672597
min           1.000000
25%          29.370000
50%         447.830000
75%         897.560000
max        4999.750000
Name: total_weight_kg, dtype: float64


In [18]:

# Diagnóstico de coherencia de los viajes

print("Cantidad de viajes:", len(trips_df))

print("\nEstados de los viajes:")
print(trips_df["status"].value_counts())

print("\nDuración de los viajes (horas):")
duraciones = (
    trips_df["arrival_datetime"]
    - trips_df["departure_datetime"]
).dt.total_seconds() / 3600

print(duraciones.describe())

print("\nConsumo de combustible (litros):")
print(trips_df["fuel_consumed_liters"].describe())

print("\nPeso transportado (kg):")
print(trips_df["total_weight_kg"].describe())


Cantidad de viajes: 100000

Estados de los viajes:
status
completed    100000
Name: count, dtype: int64

Duración de los viajes (horas):
count    100000.000000
mean          8.488980
std           4.023573
min           2.000000
25%           5.000000
50%           8.000000
75%          12.000000
max          15.000000
dtype: float64

Consumo de combustible (litros):
count    100000.000000
mean         79.875499
std          40.494210
min          10.010000
25%          44.750000
50%          79.830000
75%         114.990000
max         150.000000
Name: fuel_consumed_liters, dtype: float64

Peso transportado (kg):
count    100000.000000
mean        818.775466
std        1135.672597
min           1.000000
25%          29.370000
50%         447.830000
75%         897.560000
max        4999.750000
Name: total_weight_kg, dtype: float64


In [19]:

# Validación de los 100.000 viajes

assert len(trips_df) == 100000
assert trips_df.notna().all().all()

assert (
    trips_df["arrival_datetime"] >
    trips_df["departure_datetime"]
).all()

assert trips_df["vehicle_id"].between(1, 200).all()
assert trips_df["driver_id"].between(1, 400).all()
assert trips_df["route_id"].between(1, 50).all()

assert (trips_df["fuel_consumed_liters"] > 0).all()
assert (trips_df["total_weight_kg"] > 0).all()

capacidades = vehicles_df["capacity_kg"].to_numpy()
limites = capacidades[trips_df["vehicle_id"].to_numpy() - 1]

assert (trips_df["total_weight_kg"].to_numpy() <= limites).all()

print("✅ Los 100.000 viajes pasaron los controles básicos.")
print("✅ No hay valores nulos.")
print("✅ Fechas, identificadores y capacidades validados.")



✅ Los 100.000 viajes pasaron los controles básicos.
✅ No hay valores nulos.
✅ Fechas, identificadores y capacidades validados.


In [20]:
# Verificación previa a la generación de entregas

print("Viajes disponibles:", len(trips_df))
print("Columnas de viajes:", trips_df.columns.tolist())

assert len(trips_df) == 100000

assert (trips_df["total_weight_kg"] > 0).all()

print("✅ Viajes disponibles para generar entregas.")

Viajes disponibles: 100000
Columnas de viajes: ['vehicle_id', 'driver_id', 'route_id', 'departure_datetime', 'arrival_datetime', 'fuel_consumed_liters', 'total_weight_kg', 'status']
✅ Viajes disponibles para generar entregas.


In [21]:
import pandas as pd
import random
from datetime import timedelta

random.seed(45)

entregas = []

for i in range(1, 11):
    # Seleccionamos un viaje existente
    trip_id = random.randint(1, len(trips_df))
    viaje = trips_df.iloc[trip_id - 1]

    fecha_programada = viaje["arrival_datetime"]
    entregada = random.random() < 0.85

    fecha_entrega = (
        fecha_programada + timedelta(hours=random.randint(0, 6))
        if entregada else None
    )

    entrega = {
        "trip_id": trip_id,
        "tracking_number": f"TRK{i:08d}",
        "customer_name": f"Cliente {i}",
        "delivery_address": f"Calle {random.randint(1, 9999)}, Argentina",
        "package_weight_kg": round(
            random.uniform(0.5, min(30, float(viaje["total_weight_kg"]))), 2
        ),
        "scheduled_datetime": fecha_programada,
        "delivered_datetime": fecha_entrega,
        "delivery_status": "delivered" if entregada else "pending",
        "recipient_signature": entregada
    }

    entregas.append(entrega)

deliveries_test_df = pd.DataFrame(entregas)

print("✅ Entregas de prueba generadas:", len(deliveries_test_df))
display(deliveries_test_df)

✅ Entregas de prueba generadas: 10


,trip_id,tracking_number,customer_name,delivery_address,package_weight_kg,scheduled_datetime,delivered_datetime,delivery_status,recipient_signature
0,35636,TRK00000001,Cliente 1,"Calle 1342, Argentina",9.44,2026-05-06 13:00:00,2026-05-06 15:00:00,delivered,True
1,2831,TRK00000002,Cliente 2,"Calle 1882, Argentina",8.87,2026-09-26 17:00:00,2026-09-26 17:00:00,delivered,True
2,15429,TRK00000003,Cliente 3,"Calle 586, Argentina",27.02,2026-06-19 20:00:00,NaT,pending,False
3,8734,TRK00000004,Cliente 4,"Calle 2526, Argentina",6.72,2026-05-05 01:00:00,2026-05-05 07:00:00,delivered,True
4,99711,TRK00000005,Cliente 5,"Calle 6687, Argentina",4.71,2026-08-23 15:00:00,2026-08-23 15:00:00,delivered,True
5,7497,TRK00000006,Cliente 6,"Calle 2822, Argentina",5.16,2026-02-10 05:00:00,NaT,pending,False
6,46733,TRK00000007,Cliente 7,"Calle 1537, Argentina",12.64,2026-01-02 12:00:00,2026-01-02 14:00:00,delivered,True
7,2784,TRK00000008,Cliente 8,"Calle 6337, Argentina",9.48,2026-01-19 12:00:00,2026-01-19 14:00:00,delivered,True
8,97754,TRK00000009,Cliente 9,"Calle 4167, Argentina",18.00,2026-02-17 22:00:00,2026-02-18 02:00:00,delivered,True
9,7847,TRK00000010,Cliente 10,"Calle 3436, Argentina",10.86,2026-04-04 22:00:00,2026-04-05 00:00:00,delivered,True


In [22]:
# Controles de calidad de las entregas

assert len(deliveries_test_df) == 10

assert deliveries_test_df["tracking_number"].is_unique

assert deliveries_test_df["trip_id"].between(1, 100000).all()

assert (deliveries_test_df["package_weight_kg"] > 0).all()

assert deliveries_test_df["delivery_status"].isin(
    ["delivered", "pending"]
).all()

# Verificar coherencia de las fechas
entregadas = deliveries_test_df["delivery_status"] == "delivered"
pendientes = deliveries_test_df["delivery_status"] == "pending"

assert deliveries_test_df.loc[
    entregadas, "delivered_datetime"
].notna().all()

assert deliveries_test_df.loc[
    pendientes, "delivered_datetime"
].isna().all()

assert (
    deliveries_test_df.loc[entregadas, "delivered_datetime"]
    >= deliveries_test_df.loc[entregadas, "scheduled_datetime"]
).all()

# Verificar firmas
assert deliveries_test_df.loc[
    entregadas, "recipient_signature"
].all()

assert (~deliveries_test_df.loc[
    pendientes, "recipient_signature"
]).all()

print("✅ Las 10 entregas pasaron los controles de calidad.")
print("✅ Identificadores, fechas, estados y firmas correctos.")

✅ Las 10 entregas pasaron los controles de calidad.
✅ Identificadores, fechas, estados y firmas correctos.


In [23]:

import pandas as pd
import numpy as np

rng = np.random.default_rng(45)
n = 400_000

# Relacionar cada entrega con un viaje generado
trip_ids = rng.integers(1, len(trips_df) + 1, size=n)
viajes = trips_df.iloc[trip_ids - 1].reset_index(drop=True)

# Pesos positivos, sin superar el peso de su viaje individualmente
pesos = np.minimum(
    rng.uniform(0.5, 30, size=n),
    viajes["total_weight_kg"].to_numpy()
)

# Estados de las entregas
entregadas = rng.random(n) < 0.85
fechas_programadas = pd.to_datetime(viajes["arrival_datetime"])
demoras = pd.to_timedelta(rng.integers(0, 7, size=n), unit="h")
fechas_entrega = (fechas_programadas + demoras).where(entregadas, pd.NaT)

deliveries_df = pd.DataFrame({
    "trip_id": trip_ids,
    "tracking_number": [f"TRK{i:08d}" for i in range(1, n + 1)],
    "customer_name": [f"Cliente {i}" for i in range(1, n + 1)],
    "delivery_address": [
        f"Calle {numero}, Argentina"
        for numero in rng.integers(1, 10000, size=n)
    ],
    "package_weight_kg": np.round(pesos, 2),
    "scheduled_datetime": fechas_programadas,
    "delivered_datetime": fechas_entrega,
    "delivery_status": np.where(entregadas, "delivered", "pending"),
    "recipient_signature": entregadas
})

print("✅ Entregas generadas:", len(deliveries_df))
display(deliveries_df.head(10))


✅ Entregas generadas: 400000


,trip_id,tracking_number,customer_name,delivery_address,package_weight_kg,scheduled_datetime,delivered_datetime,delivery_status,recipient_signature
0,92454,TRK00000001,Cliente 1,"Calle 7261, Argentina",21.23,2026-10-15 23:00:00,NaT,pending,False
1,57314,TRK00000002,Cliente 2,"Calle 6597, Argentina",22.99,2026-04-23 10:00:00,2026-04-23 16:00:00,delivered,True
2,73161,TRK00000003,Cliente 3,"Calle 8190, Argentina",6.49,2026-10-03 02:00:00,2026-10-03 03:00:00,delivered,True
3,52850,TRK00000004,Cliente 4,"Calle 1763, Argentina",9.82,2026-04-05 22:00:00,2026-04-06 01:00:00,delivered,True
4,53184,TRK00000005,Cliente 5,"Calle 1969, Argentina",2.69,2026-03-07 09:00:00,2026-03-07 15:00:00,delivered,True
5,76366,TRK00000006,Cliente 6,"Calle 6380, Argentina",20.26,2026-06-26 12:00:00,2026-06-26 18:00:00,delivered,True
6,70601,TRK00000007,Cliente 7,"Calle 6989, Argentina",1.42,2026-09-01 12:00:00,2026-09-01 18:00:00,delivered,True
7,81170,TRK00000008,Cliente 8,"Calle 3241, Argentina",15.53,2026-07-21 09:00:00,2026-07-21 15:00:00,delivered,True
8,56263,TRK00000009,Cliente 9,"Calle 8396, Argentina",19.64,2026-04-30 18:00:00,2026-04-30 21:00:00,delivered,True
9,51023,TRK00000010,Cliente 10,"Calle 1238, Argentina",23.96,2026-09-27 13:00:00,2026-09-27 13:00:00,delivered,True


### 3.5. Generación de entregas

Se generan 400.000 entregas sintéticas asociadas a los viajes de FleetLogix.

Cada entrega contiene un número de seguimiento único, un cliente, una dirección, el peso del paquete, las fechas programada y efectiva de entrega, el estado y la confirmación de recepción.

**Objetivo:** simular la distribución de paquetes y disponer de información para analizar el cumplimiento de las entregas y la trazabilidad logística.

Se realizan controles de calidad para verificar la cantidad de registros, la unicidad de los números de seguimiento, la asociación con viajes válidos, los pesos positivos y la coherencia entre el estado y la fecha de entrega.

In [24]:

# Controles de calidad de las 400.000 entregas

assert len(deliveries_df) == 400_000
assert deliveries_df["tracking_number"].is_unique
assert deliveries_df["trip_id"].between(1, len(trips_df)).all()
assert (deliveries_df["package_weight_kg"] > 0).all()

entregadas = deliveries_df["delivery_status"] == "delivered"
pendientes = deliveries_df["delivery_status"] == "pending"

assert (entregadas | pendientes).all()
assert deliveries_df.loc[entregadas, "delivered_datetime"].notna().all()
assert deliveries_df.loc[pendientes, "delivered_datetime"].isna().all()

assert (
    deliveries_df.loc[entregadas, "delivered_datetime"]
    >= deliveries_df.loc[entregadas, "scheduled_datetime"]
).all()

assert deliveries_df["recipient_signature"].eq(entregadas).all()

print("✅ Las 400.000 entregas pasaron los controles básicos.")
print("✅ Identificadores, fechas, estados y firmas correctos.")


✅ Las 400.000 entregas pasaron los controles básicos.
✅ Identificadores, fechas, estados y firmas correctos.


In [25]:

# Verificar si ya generamos los mantenimientos

if "maintenance_df" in globals():
    print("Mantenimientos generados:", len(maintenance_df))
    display(maintenance_df.head())
else:
    print("Todavía no generamos los mantenimientos.")


Todavía no generamos los mantenimientos.


### 3.6. Generación de mantenimientos

Se generan 5.000 registros sintéticos de mantenimiento para representar las tareas de conservación y reparación de la flota de FleetLogix.

Cada mantenimiento incluye el vehículo asociado, la fecha de realización, el tipo de servicio, la descripción, el costo, la fecha del próximo mantenimiento y el taller responsable.

**Objetivo:** disponer de información para analizar los costos de mantenimiento, planificar servicios preventivos y evaluar el estado operativo de los vehículos.

La generación contempla diferentes tipos de mantenimiento, rangos de costos e intervalos entre servicios.

In [26]:

import pandas as pd
import random
from datetime import date, timedelta

random.seed(45)

tipos_mantenimiento = {
    "Cambio de aceite": (15000, 65000, 90),
    "Revision de frenos": (25000, 140000, 180),
    "Cambio de neumaticos": (80000, 350000, 365),
    "Mantenimiento preventivo": (20000, 120000, 180),
    "Reparacion de motor": (150000, 950000, 365),
    "Revision electrica": (18000, 95000, 180)
}

talleres = [
    "Taller Central FleetLogix",
    "Servicio Tecnico Norte",
    "Mecanica Integral",
    "Taller Ruta Segura",
    "Servicio Automotor Sur"
]

mantenimientos = []

for i in range(5000):
    tipo = random.choice(list(tipos_mantenimiento))
    costo_min, costo_max, intervalo = tipos_mantenimiento[tipo]

    fecha = date(2026, 1, 1) + timedelta(
        days=random.randint(0, 300)
    )

    mantenimiento = {
        "vehicle_id": random.randint(1, 200),
        "maintenance_date": fecha,
        "maintenance_type": tipo,
        "description": f"{tipo} realizado al vehiculo.",
        "cost": round(random.uniform(costo_min, costo_max), 2),
        "next_maintenance_date": fecha + timedelta(days=intervalo),
        "performed_by": random.choice(talleres)
    }

    mantenimientos.append(mantenimiento)

maintenance_df = pd.DataFrame(mantenimientos)

print("✅ Mantenimientos generados:", len(maintenance_df))
display(maintenance_df.head(10))


✅ Mantenimientos generados: 5000


,vehicle_id,maintenance_date,maintenance_type,description,cost,next_maintenance_date,performed_by
0,125,2026-08-02,Cambio de neumaticos,Cambio de neumaticos realizado al vehiculo.,149568.37,2027-08-02,Mecanica Integral
1,19,2026-01-12,Cambio de neumaticos,Cambio de neumaticos realizado al vehiculo.,210749.10,2027-01-12,Taller Central FleetLogix
2,80,2026-03-02,Cambio de neumaticos,Cambio de neumaticos realizado al vehiculo.,310963.39,2027-03-02,Taller Central FleetLogix
3,35,2026-02-04,Reparacion de motor,Reparacion de motor realizado al vehiculo.,367268.30,2027-02-04,Servicio Tecnico Norte
4,20,2026-08-08,Revision de frenos,Revision de frenos realizado al vehiculo.,95485.53,2027-02-04,Servicio Tecnico Norte
5,9,2026-01-30,Revision electrica,Revision electrica realizado al vehiculo.,31260.26,2026-07-29,Mecanica Integral
6,25,2026-06-13,Cambio de neumaticos,Cambio de neumaticos realizado al vehiculo.,191076.94,2027-06-13,Taller Central FleetLogix
7,100,2026-06-12,Cambio de neumaticos,Cambio de neumaticos realizado al vehiculo.,268779.36,2027-06-12,Servicio Automotor Sur
8,162,2026-10-02,Revision electrica,Revision electrica realizado al vehiculo.,74367.37,2027-03-31,Mecanica Integral
9,127,2026-01-31,Reparacion de motor,Reparacion de motor realizado al vehiculo.,877689.29,2027-01-31,Servicio Tecnico Norte


In [27]:

assert len(maintenance_df) == 5000
assert maintenance_df.notna().all().all()

assert maintenance_df["vehicle_id"].between(1, 200).all()
assert (maintenance_df["cost"] > 0).all()

assert (
    maintenance_df["next_maintenance_date"]
    > maintenance_df["maintenance_date"]
).all()

assert maintenance_df["maintenance_type"].str.len().le(50).all()
assert maintenance_df["performed_by"].str.len().le(200).all()

print("✅ Los 5.000 mantenimientos pasaron los controles básicos.")
print("✅ Vehículos, fechas, costos y textos correctos.")


✅ Los 5.000 mantenimientos pasaron los controles básicos.
✅ Vehículos, fechas, costos y textos correctos.


In [28]:
tablas = {
    "Vehículos": vehicles_df,
    "Conductores": drivers_df,
    "Rutas": routes_df,
    "Viajes": trips_df,
    "Entregas": deliveries_df,
    "Mantenimientos": maintenance_df
}

for nombre, df in tablas.items():
    print(f"{nombre}: {len(df):,} registros")

print(f"\nTOTAL: {sum(len(df) for df in tablas.values()):,} registros")

Vehículos: 200 registros
Conductores: 400 registros
Rutas: 50 registros
Viajes: 100,000 registros
Entregas: 400,000 registros
Mantenimientos: 5,000 registros

TOTAL: 505,650 registros


In [29]:
assert trips_df["vehicle_id"].between(1, 200).all()
assert trips_df["driver_id"].between(1, 400).all()
assert trips_df["route_id"].between(1, 50).all()

assert deliveries_df["trip_id"].between(1, 100000).all()
assert maintenance_df["vehicle_id"].between(1, 200).all()

print("✅ Relaciones básicas verificadas.")

✅ Relaciones básicas verificadas.


## 4. Validación de calidad e integridad de datos

### 4.1. Integridad referencial

Se verifica que las relaciones entre las seis tablas de FleetLogix sean consistentes.

Se controlan las siguientes relaciones:

- Viajes → Vehículos.
- Viajes → Conductores.
- Viajes → Rutas.
- Entregas → Viajes.
- Mantenimientos → Vehículos.

**Resultado:** las cinco relaciones superaron los controles básicos de integridad referencial realizados sobre los DataFrames de Pandas.

Estas validaciones permiten comprobar que los identificadores relacionados corresponden a registros válidos.

In [30]:
assert set(trips_df["vehicle_id"]).issubset(
    set(range(1, len(vehicles_df) + 1))
)

assert set(trips_df["driver_id"]).issubset(
    set(range(1, len(drivers_df) + 1))
)

assert set(trips_df["route_id"]).issubset(
    set(range(1, len(routes_df) + 1))
)

assert set(deliveries_df["trip_id"]).issubset(
    set(range(1, len(trips_df) + 1))
)

assert set(maintenance_df["vehicle_id"]).issubset(
    set(range(1, len(vehicles_df) + 1))
)

print("✅ Integridad referencial básica verificada.")

✅ Integridad referencial básica verificada.


In [31]:
tablas = {
    "vehicles": vehicles_df,
    "drivers": drivers_df,
    "routes": routes_df,
    "trips": trips_df,
    "deliveries": deliveries_df,
    "maintenance": maintenance_df
}

for nombre, df in tablas.items():
    print(f"\n{nombre}: {len(df):,} registros")
    print("Columnas:", df.columns.tolist())


vehicles: 200 registros
Columnas: ['license_plate', 'vehicle_type', 'capacity_kg', 'fuel_type', 'acquisition_date', 'status']

drivers: 400 registros
Columnas: ['employee_code', 'first_name', 'last_name', 'license_number', 'license_expiry', 'phone', 'hire_date', 'status']

routes: 50 registros
Columnas: ['route_code', 'origin_city', 'destination_city', 'distance_km', 'estimated_duration_hours', 'toll_cost']

trips: 100,000 registros
Columnas: ['vehicle_id', 'driver_id', 'route_id', 'departure_datetime', 'arrival_datetime', 'fuel_consumed_liters', 'total_weight_kg', 'status']

deliveries: 400,000 registros
Columnas: ['trip_id', 'tracking_number', 'customer_name', 'delivery_address', 'package_weight_kg', 'scheduled_datetime', 'delivered_datetime', 'delivery_status', 'recipient_signature']

maintenance: 5,000 registros
Columnas: ['vehicle_id', 'maintenance_date', 'maintenance_type', 'description', 'cost', 'next_maintenance_date', 'performed_by']


In [32]:

# Preparar los vehículos para PostgreSQL
vehicles_to_load = vehicles_df.copy()

print("Vehículos preparados:", len(vehicles_to_load))
print("Columnas:", vehicles_to_load.columns.tolist())

display(vehicles_to_load.head())


Vehículos preparados: 200
Columnas: ['license_plate', 'vehicle_type', 'capacity_kg', 'fuel_type', 'acquisition_date', 'status']


,license_plate,vehicle_type,capacity_kg,fuel_type,acquisition_date,status
0,FLX0001,motorcycle,30,gasoline,2023-08-27,active
1,FLX0002,van,850,electric,2025-08-03,active
2,FLX0003,motorcycle,30,electric,2026-05-19,active
3,FLX0004,van,850,diesel,2021-01-26,active
4,FLX0005,van,850,electric,2020-08-16,active


In [10]:

import psycopg2
from getpass import getpass

password = getpass("Contraseña de PostgreSQL: ")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="fleetlogix",
    user="fleetlogix_user",
    password=password
)

print("✅ Conexión establecida correctamente.")

with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM public.vehicles")
    print("Vehículos actualmente en PostgreSQL:", cursor.fetchone()[0])


Contraseña de PostgreSQL:  ········


✅ Conexión establecida correctamente.
Vehículos actualmente en PostgreSQL: 200


In [34]:

from psycopg2.extras import execute_values

columnas = [
    "license_plate",
    "vehicle_type",
    "capacity_kg",
    "fuel_type",
    "acquisition_date",
    "status"
]

datos = [
    tuple(fila)
    for fila in vehicles_to_load[columnas].itertuples(
        index=False, name=None
    )
]

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM public.vehicles")
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla ya contiene registros. No se insertó nada.")
        else:
            execute_values(
                cursor,
                """
                INSERT INTO public.vehicles
                (license_plate, vehicle_type, capacity_kg,
                 fuel_type, acquisition_date, status)
                VALUES %s
                """,
                datos
            )
            conn.commit()
            print("✅ Vehículos insertados:", len(datos))

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


✅ Vehículos insertados: 200


In [35]:

with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM public.vehicles")
    total = cursor.fetchone()[0]

print("Vehículos guardados en PostgreSQL:", total)

assert total == 200, "La cantidad no coincide"
print("✅ Verificación exitosa")


Vehículos guardados en PostgreSQL: 200
✅ Verificación exitosa


In [36]:

# Preparar conductores para PostgreSQL
drivers_to_load = drivers_df.copy()

print("Conductores preparados:", len(drivers_to_load))
print("Columnas:", drivers_to_load.columns.tolist())

display(drivers_to_load.head())


Conductores preparados: 400
Columnas: ['employee_code', 'first_name', 'last_name', 'license_number', 'license_expiry', 'phone', 'hire_date', 'status']


,employee_code,first_name,last_name,license_number,license_expiry,phone,hire_date,status
0,EMP0001,Luca,Gonzalez,LIC000001,2030-07-04,1124942603,2026-06-24,active
1,EMP0002,Victoria,Diaz,LIC000002,2028-04-17,1139958838,2025-03-12,active
2,EMP0003,Abril,Peralta,LIC000003,2030-09-18,1183197857,2025-10-13,active
3,EMP0004,Uma,Rodriguez,LIC000004,2027-02-06,1113999315,2025-09-16,active
4,EMP0005,Ciro,Gonzalez,LIC000005,2029-10-02,1190801586,2026-06-18,active


In [37]:

from psycopg2.extras import execute_values

columnas_drivers = [
    "employee_code",
    "first_name",
    "last_name",
    "license_number",
    "license_expiry",
    "phone",
    "hire_date",
    "status"
]

datos_drivers = list(
    drivers_to_load[columnas_drivers].itertuples(
        index=False, name=None
    )
)

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM public.drivers")
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla ya contiene registros. No se insertó nada.")
        else:
            execute_values(
                cursor,
                """
                INSERT INTO public.drivers
                (employee_code, first_name, last_name,
                 license_number, license_expiry, phone,
                 hire_date, status)
                VALUES %s
                """,
                datos_drivers
            )

            conn.commit()
            print("✅ Conductores insertados:", len(datos_drivers))

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


✅ Conductores insertados: 400


In [38]:

with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM public.drivers")
    total_drivers = cursor.fetchone()[0]

print("Conductores guardados en PostgreSQL:", total_drivers)

assert total_drivers == 400, "La cantidad no coincide"
print("✅ Verificación exitosa")


Conductores guardados en PostgreSQL: 400
✅ Verificación exitosa


In [39]:

from psycopg2.extras import execute_values

columnas_routes = [
    "route_code",
    "origin_city",
    "destination_city",
    "distance_km",
    "estimated_duration_hours",
    "toll_cost"
]

datos_routes = list(
    routes_df[columnas_routes].itertuples(
        index=False, name=None
    )
)

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM public.routes")
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla ya contiene registros. No se insertó nada.")
        else:
            execute_values(
                cursor,
                """
                INSERT INTO public.routes
                (route_code, origin_city, destination_city,
                 distance_km, estimated_duration_hours, toll_cost)
                VALUES %s
                """,
                datos_routes
            )

            conn.commit()
            print("✅ Rutas insertadas:", len(datos_routes))

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


✅ Rutas insertadas: 50


In [40]:

tablas = {
    "vehicles": 200,
    "drivers": 400,
    "routes": 50
}

with conn.cursor() as cursor:
    for tabla, esperado in tablas.items():
        cursor.execute(
            f"""
            SELECT COUNT(*), MIN({tabla[:-1] + '_id' if tabla != 'vehicles' else 'vehicle_id'}),
                   MAX({tabla[:-1] + '_id' if tabla != 'vehicles' else 'vehicle_id'})
            FROM public.{tabla}
            """
        )
        cantidad, minimo, maximo = cursor.fetchone()
        print(f"{tabla}: cantidad={cantidad}, ID mínimo={minimo}, ID máximo={maximo}")

        assert cantidad == esperado, f"Cantidad incorrecta en {tabla}"
        assert minimo == 1 and maximo == esperado, (
            f"Los identificadores de {tabla} necesitan revisión"
        )

print("✅ Tablas e identificadores verificados")


vehicles: cantidad=200, ID mínimo=1, ID máximo=200
drivers: cantidad=400, ID mínimo=1, ID máximo=400
routes: cantidad=50, ID mínimo=1, ID máximo=50
✅ Tablas e identificadores verificados


In [41]:

import pandas as pd

print("Viajes preparados:", len(trips_df))
print("Columnas:", trips_df.columns.tolist())

print("\nValores faltantes:")
print(trips_df.isnull().sum())

print("\nRangos de identificadores:")
for columna, maximo in [
    ("vehicle_id", 200),
    ("driver_id", 400),
    ("route_id", 50)
]:
    minimo_real = trips_df[columna].min()
    maximo_real = trips_df[columna].max()

    print(f"{columna}: {minimo_real} a {maximo_real}")

    assert trips_df[columna].between(1, maximo).all(), (
        f"❌ Identificadores incorrectos en {columna}"
    )

assert len(trips_df) == 100000, "❌ Cantidad incorrecta de viajes"

print("\n✅ Viajes preparados y relaciones verificadas")


Viajes preparados: 100000
Columnas: ['vehicle_id', 'driver_id', 'route_id', 'departure_datetime', 'arrival_datetime', 'fuel_consumed_liters', 'total_weight_kg', 'status']

Valores faltantes:
vehicle_id              0
driver_id               0
route_id                0
departure_datetime      0
arrival_datetime        0
fuel_consumed_liters    0
total_weight_kg         0
status                  0
dtype: int64

Rangos de identificadores:
vehicle_id: 1 a 200
driver_id: 1 a 400
route_id: 1 a 50

✅ Viajes preparados y relaciones verificadas


In [42]:

from psycopg2.extras import execute_values
import pandas as pd

columnas_trips = [
    "vehicle_id", "driver_id", "route_id",
    "departure_datetime", "arrival_datetime",
    "fuel_consumed_liters", "total_weight_kg", "status"
]

trips_to_load = trips_df[columnas_trips].copy()
trips_to_load = trips_to_load.astype(object).where(
    pd.notna(trips_to_load), None
)

sql = """
INSERT INTO public.trips (
    vehicle_id, driver_id, route_id,
    departure_datetime, arrival_datetime,
    fuel_consumed_liters, total_weight_kg, status
) VALUES %s
"""

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM public.trips")
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla trips ya contiene registros.")
            print("No se insertaron datos.")
        else:
            total = len(trips_to_load)
            lote = 5000

            for inicio in range(0, total, lote):
                bloque = trips_to_load.iloc[inicio:inicio + lote]

                datos = list(
                    bloque.itertuples(index=False, name=None)
                )

                execute_values(
                    cursor, sql, datos, page_size=1000
                )

                print(
                    f"Preparados para confirmar: "
                    f"{min(inicio + lote, total):,} / {total:,}"
                )

            conn.commit()
            print("✅ Viajes insertados:", total)

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


Preparados para confirmar: 5,000 / 100,000
Preparados para confirmar: 10,000 / 100,000
Preparados para confirmar: 15,000 / 100,000
Preparados para confirmar: 20,000 / 100,000
Preparados para confirmar: 25,000 / 100,000
Preparados para confirmar: 30,000 / 100,000
Preparados para confirmar: 35,000 / 100,000
Preparados para confirmar: 40,000 / 100,000
Preparados para confirmar: 45,000 / 100,000
Preparados para confirmar: 50,000 / 100,000
Preparados para confirmar: 55,000 / 100,000
Preparados para confirmar: 60,000 / 100,000
Preparados para confirmar: 65,000 / 100,000
Preparados para confirmar: 70,000 / 100,000
Preparados para confirmar: 75,000 / 100,000
Preparados para confirmar: 80,000 / 100,000
Preparados para confirmar: 85,000 / 100,000
Preparados para confirmar: 90,000 / 100,000
Preparados para confirmar: 95,000 / 100,000
Preparados para confirmar: 100,000 / 100,000
✅ Viajes insertados: 100000


In [43]:

with conn.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*), MIN(trip_id), MAX(trip_id)
        FROM public.trips
    """)
    cantidad, minimo, maximo = cursor.fetchone()

print("Viajes guardados:", cantidad)
print("ID mínimo:", minimo)
print("ID máximo:", maximo)

assert cantidad == 100000, "❌ Cantidad incorrecta"
assert minimo == 1 and maximo == 100000, (
    "❌ Los identificadores necesitan revisión"
)

print("✅ Viajes e identificadores verificados")


Viajes guardados: 100000
ID mínimo: 1
ID máximo: 100000
✅ Viajes e identificadores verificados


In [44]:

import pandas as pd

print("Entregas preparadas:", len(deliveries_df))
print("\nColumnas:")
print(deliveries_df.columns.tolist())

print("\nValores faltantes:")
print(deliveries_df.isnull().sum())

print("\nRango de trip_id:")
print("Mínimo:", deliveries_df["trip_id"].min())
print("Máximo:", deliveries_df["trip_id"].max())

assert len(deliveries_df) == 400000, (
    "❌ Cantidad incorrecta de entregas"
)

assert deliveries_df["trip_id"].between(
    1, 100000
).all(), "❌ Hay viajes inexistentes"

print("\nTipos de datos:")
print(deliveries_df.dtypes)

print("\n✅ Entregas y referencias preparadas")


Entregas preparadas: 400000

Columnas:
['trip_id', 'tracking_number', 'customer_name', 'delivery_address', 'package_weight_kg', 'scheduled_datetime', 'delivered_datetime', 'delivery_status', 'recipient_signature']

Valores faltantes:
trip_id                    0
tracking_number            0
customer_name              0
delivery_address           0
package_weight_kg          0
scheduled_datetime         0
delivered_datetime     60180
delivery_status            0
recipient_signature        0
dtype: int64

Rango de trip_id:
Mínimo: 1
Máximo: 100000

Tipos de datos:
trip_id                         int64
tracking_number                   str
customer_name                     str
delivery_address                  str
package_weight_kg             float64
scheduled_datetime     datetime64[us]
delivered_datetime     datetime64[us]
delivery_status                   str
recipient_signature              bool
dtype: object

✅ Entregas y referencias preparadas


In [45]:

from psycopg2.extras import execute_values
import pandas as pd

columnas_deliveries = [
    "trip_id", "tracking_number", "customer_name",
    "delivery_address", "package_weight_kg",
    "scheduled_datetime", "delivered_datetime",
    "delivery_status", "recipient_signature"
]

sql_deliveries = """
INSERT INTO public.deliveries (
    trip_id, tracking_number, customer_name,
    delivery_address, package_weight_kg,
    scheduled_datetime, delivered_datetime,
    delivery_status, recipient_signature
) VALUES %s
"""

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM public.deliveries")
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla ya contiene registros.")
            print("No se insertó nada.")
        else:
            total = len(deliveries_df)
            lote = 5000

            for inicio in range(0, total, lote):
                bloque = deliveries_df[
                    columnas_deliveries
                ].iloc[inicio:inicio + lote].copy()

                # Convertir fechas vacías a NULL
                bloque = bloque.astype(object).where(
                    pd.notna(bloque), None
                )

                datos = list(
                    bloque.itertuples(index=False, name=None)
                )

                execute_values(
                    cursor,
                    sql_deliveries,
                    datos,
                    page_size=1000
                )

                print(
                    f"Preparados para confirmar: "
                    f"{min(inicio + lote, total):,} / {total:,}",
                    flush=True
                )

            conn.commit()
            print("✅ Entregas insertadas:", total)

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


Preparados para confirmar: 5,000 / 400,000
Preparados para confirmar: 10,000 / 400,000
Preparados para confirmar: 15,000 / 400,000
Preparados para confirmar: 20,000 / 400,000
Preparados para confirmar: 25,000 / 400,000
Preparados para confirmar: 30,000 / 400,000
Preparados para confirmar: 35,000 / 400,000
Preparados para confirmar: 40,000 / 400,000
Preparados para confirmar: 45,000 / 400,000
Preparados para confirmar: 50,000 / 400,000
Preparados para confirmar: 55,000 / 400,000
Preparados para confirmar: 60,000 / 400,000
Preparados para confirmar: 65,000 / 400,000
Preparados para confirmar: 70,000 / 400,000
Preparados para confirmar: 75,000 / 400,000
Preparados para confirmar: 80,000 / 400,000
Preparados para confirmar: 85,000 / 400,000
Preparados para confirmar: 90,000 / 400,000
Preparados para confirmar: 95,000 / 400,000
Preparados para confirmar: 100,000 / 400,000
Preparados para confirmar: 105,000 / 400,000
Preparados para confirmar: 110,000 / 400,000
Preparados para confirmar: 115

In [46]:

with conn.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*), COUNT(DISTINCT tracking_number),
               COUNT(*) FILTER (WHERE delivered_datetime IS NULL)
        FROM public.deliveries
    """)
    total, tracking_unicos, fechas_vacias = cursor.fetchone()

print("Entregas guardadas:", total)
print("Números de seguimiento únicos:", tracking_unicos)
print("Fechas de entrega vacías:", fechas_vacias)

assert total == 400000, "❌ Cantidad incorrecta"
assert tracking_unicos == 400000, "❌ Hay números de seguimiento duplicados"
assert fechas_vacias == 60180, "❌ No coincide la cantidad de fechas vacías"

print("✅ Entregas verificadas correctamente")


Entregas guardadas: 400000
Números de seguimiento únicos: 400000
Fechas de entrega vacías: 60180
✅ Entregas verificadas correctamente


In [47]:

print("Mantenimientos preparados:", len(maintenance_df))

print("\nColumnas:")
print(maintenance_df.columns.tolist())

print("\nValores faltantes:")
print(maintenance_df.isnull().sum())

print("\nRango de vehicle_id:")
print("Mínimo:", maintenance_df["vehicle_id"].min())
print("Máximo:", maintenance_df["vehicle_id"].max())

assert len(maintenance_df) == 5000, (
    "❌ Cantidad incorrecta de mantenimientos"
)

assert maintenance_df["vehicle_id"].between(
    1, 200
).all(), "❌ Hay vehículos inexistentes"

print("\nTipos de datos:")
print(maintenance_df.dtypes)

print("\n✅ Mantenimientos preparados correctamente")


Mantenimientos preparados: 5000

Columnas:
['vehicle_id', 'maintenance_date', 'maintenance_type', 'description', 'cost', 'next_maintenance_date', 'performed_by']

Valores faltantes:
vehicle_id               0
maintenance_date         0
maintenance_type         0
description              0
cost                     0
next_maintenance_date    0
performed_by             0
dtype: int64

Rango de vehicle_id:
Mínimo: 1
Máximo: 200

Tipos de datos:
vehicle_id                 int64
maintenance_date          object
maintenance_type             str
description                  str
cost                     float64
next_maintenance_date     object
performed_by                 str
dtype: object

✅ Mantenimientos preparados correctamente


In [48]:

from psycopg2.extras import execute_values
import pandas as pd

columnas_maintenance = [
    "vehicle_id",
    "maintenance_date",
    "maintenance_type",
    "description",
    "cost",
    "next_maintenance_date",
    "performed_by"
]

maintenance_to_load = (
    maintenance_df[columnas_maintenance]
    .copy()
    .astype(object)
)

maintenance_to_load = maintenance_to_load.where(
    pd.notna(maintenance_to_load), None
)

datos_maintenance = list(
    maintenance_to_load.itertuples(
        index=False, name=None
    )
)

try:
    with conn.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM public.maintenance"
        )
        cantidad_actual = cursor.fetchone()[0]

        if cantidad_actual != 0:
            print("⚠️ La tabla ya contiene registros.")
            print("No se insertó nada.")
        else:
            execute_values(
                cursor,
                """
                INSERT INTO public.maintenance (
                    vehicle_id,
                    maintenance_date,
                    maintenance_type,
                    description,
                    cost,
                    next_maintenance_date,
                    performed_by
                )
                VALUES %s
                """,
                datos_maintenance,
                page_size=1000
            )

            conn.commit()
            print(
                "✅ Mantenimientos insertados:",
                len(datos_maintenance)
            )

except Exception as error:
    conn.rollback()
    print("❌ Error:", error)


✅ Mantenimientos insertados: 5000


In [49]:

tablas = {
    "vehicles": 200,
    "drivers": 400,
    "routes": 50,
    "trips": 100000,
    "deliveries": 400000,
    "maintenance": 5000
}

total_general = 0

with conn.cursor() as cursor:
    for tabla, esperado in tablas.items():
        cursor.execute(
            f"SELECT COUNT(*) FROM public.{tabla}"
        )
        cantidad = cursor.fetchone()[0]
        total_general += cantidad

        print(f"{tabla}: {cantidad:,} registros")

        assert cantidad == esperado, (
            f"❌ Cantidad incorrecta en {tabla}"
        )

print("\nTOTAL GENERAL:", f"{total_general:,}")

assert total_general == 505650, (
    "❌ El total general no coincide"
)

print("✅ Las seis tablas están verificadas")


vehicles: 200 registros
drivers: 400 registros
routes: 50 registros
trips: 100,000 registros
deliveries: 400,000 registros
maintenance: 5,000 registros

TOTAL GENERAL: 505,650
✅ Las seis tablas están verificadas


In [50]:

consultas = {
    "Viajes sin vehículo": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.vehicles v
        ON t.vehicle_id = v.vehicle_id
        WHERE v.vehicle_id IS NULL
    """,
    "Viajes sin conductor": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.drivers d
        ON t.driver_id = d.driver_id
        WHERE d.driver_id IS NULL
    """,
    "Viajes sin ruta": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.routes r
        ON t.route_id = r.route_id
        WHERE r.route_id IS NULL
    """,
    "Entregas sin viaje": """
        SELECT COUNT(*) FROM public.deliveries d
        LEFT JOIN public.trips t
        ON d.trip_id = t.trip_id
        WHERE t.trip_id IS NULL
    """,
    "Mantenimientos sin vehículo": """
        SELECT COUNT(*) FROM public.maintenance m
        LEFT JOIN public.vehicles v
        ON m.vehicle_id = v.vehicle_id
        WHERE v.vehicle_id IS NULL
    """
}

with conn.cursor() as cursor:
    for nombre, sql in consultas.items():
        cursor.execute(sql)
        errores = cursor.fetchone()[0]

        print(f"{nombre}: {errores}")

        assert errores == 0, (
            f"❌ Se encontraron errores: {nombre}"
        )

print("\n✅ Integridad referencial verificada")


Viajes sin vehículo: 0
Viajes sin conductor: 0
Viajes sin ruta: 0
Entregas sin viaje: 0
Mantenimientos sin vehículo: 0

✅ Integridad referencial verificada


In [51]:

consultas = {
    "Viajes sin vehículo": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.vehicles v
        ON t.vehicle_id = v.vehicle_id
        WHERE v.vehicle_id IS NULL
    """,
    "Viajes sin conductor": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.drivers d
        ON t.driver_id = d.driver_id
        WHERE d.driver_id IS NULL
    """,
    "Viajes sin ruta": """
        SELECT COUNT(*) FROM public.trips t
        LEFT JOIN public.routes r
        ON t.route_id = r.route_id
        WHERE r.route_id IS NULL
    """,
    "Entregas sin viaje": """
        SELECT COUNT(*) FROM public.deliveries d
        LEFT JOIN public.trips t
        ON d.trip_id = t.trip_id
        WHERE t.trip_id IS NULL
    """,
    "Mantenimientos sin vehículo": """
        SELECT COUNT(*) FROM public.maintenance m
        LEFT JOIN public.vehicles v
        ON m.vehicle_id = v.vehicle_id
        WHERE v.vehicle_id IS NULL
    """
}

with conn.cursor() as cursor:
    for nombre, sql in consultas.items():
        cursor.execute(sql)
        errores = cursor.fetchone()[0]

        print(f"{nombre}: {errores}")

        assert errores == 0, (
            f"❌ Se encontraron errores: {nombre}"
        )

print("\n✅ Integridad referencial verificada")


Viajes sin vehículo: 0
Viajes sin conductor: 0
Viajes sin ruta: 0
Entregas sin viaje: 0
Mantenimientos sin vehículo: 0

✅ Integridad referencial verificada


In [52]:

with conn.cursor() as cursor:
    cursor.execute("""
        SELECT
            COUNT(*) AS total_viajes,
            COUNT(*) FILTER (
                WHERE arrival_datetime IS NOT NULL
                AND arrival_datetime <= departure_datetime
            ) AS fechas_incorrectas,
            COUNT(*) FILTER (
                WHERE arrival_datetime IS NULL
            ) AS llegadas_pendientes
        FROM public.trips
    """)

    total, incorrectas, pendientes = cursor.fetchone()

print("Total de viajes:", total)
print("Fechas incorrectas:", incorrectas)
print("Llegadas pendientes:", pendientes)

assert incorrectas == 0, (
    "❌ Existen viajes con fechas inconsistentes"
)

print("✅ Consistencia temporal verificada")


Total de viajes: 100000
Fechas incorrectas: 0
Llegadas pendientes: 0
✅ Consistencia temporal verificada


## Validaciones de calidad de datos — Avance 1

En esta sección se verifica la calidad y consistencia de los datos almacenados en PostgreSQL.

### Resultados obtenidos

- Total de registros: 505.650
- Integridad referencial: correcta.
- Consistencia temporal de viajes: correcta.
- Viajes analizados: 100.000.
- Viajes con sobrecarga: 0.
- Viajes sin datos de capacidad: 0.
- Entregas analizadas: 400.000.
- Números de seguimiento vacíos: 0.
- Números de seguimiento duplicados: 0.

In [15]:
with conexion.cursor() as cursor:
    cursor.execute("""
        SELECT
            COUNT(*) AS total_viajes,
            COUNT(*) FILTER (
                WHERE t.total_weight_kg > v.capacity_kg
            ) AS viajes_sobrecargados
        FROM public.trips t
        JOIN public.vehicles v
            ON t.vehicle_id = v.vehicle_id
    """)

    total, sobrecargados = cursor.fetchone()

print("Total de viajes:", total)
print("Viajes sobrecargados:", sobrecargados)

if sobrecargados == 0:
    print("✅ Capacidad de vehículos verificada")
else:
    print("⚠️ Se detectaron viajes sobrecargados")

Total de viajes: 100000
Viajes sobrecargados: 0
✅ Capacidad de vehículos verificada


### Consultas SQL de validación

**1. Capacidad de vehículos**

```sql
SELECT
    COUNT(*) AS total_viajes,
    COUNT(*) FILTER (
        WHERE t.total_weight_kg > v.capacity_kg
    ) AS viajes_sobrecargados,
    COUNT(*) FILTER (
        WHERE t.total_weight_kg IS NULL
           OR v.capacity_kg IS NULL
    ) AS capacidades_sin_datos
FROM public.trips t
JOIN public.vehicles v
    ON t.vehicle_id = v.vehicle_id;

In [14]:
import psycopg2
from getpass import getpass

conexion = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="fleetlogix",
    user="fleetlogix_user",
    password=getpass("Contraseña: ")
)

print("✅ Conexión activa:", conexion.closed == 0)

Contraseña:  ········


✅ Conexión activa: True


In [17]:
with conexion.cursor() as cursor:
    cursor.execute("""
        SELECT
            COUNT(*) AS total_entregas,
            COUNT(*) FILTER (
                WHERE tracking_number IS NULL
                   OR TRIM(tracking_number) = ''
            ) AS numeros_vacios,
            COUNT(*) - COUNT(DISTINCT tracking_number)
                AS numeros_duplicados
        FROM public.deliveries;
    """)

    total, vacios, duplicados = cursor.fetchone()

print("Total de entregas:", total)
print("Números vacíos:", vacios)
print("Números duplicados:", duplicados)

if vacios == 0 and duplicados == 0:
    print("✅ Números de seguimiento verificados")
else:
    print("⚠️ Se encontraron problemas")

Total de entregas: 400000
Números vacíos: 0
Números duplicados: 0
✅ Números de seguimiento verificados


### 2. Validación de números de seguimiento

```sql
SELECT
    COUNT(*) AS total_entregas,
    COUNT(*) FILTER (
        WHERE tracking_number IS NULL
           OR TRIM(tracking_number) = ''
    ) AS seguimientos_vacios,
    COUNT(*) - COUNT(DISTINCT tracking_number)
        AS seguimientos_duplicados
FROM public.deliveries;
```

**Resultados:** 400.000 entregas, 0 seguimientos vacíos y 0 duplicados.

In [18]:
import json
from datetime import datetime

reporte = {
    "proyecto": "FleetLogix",
    "avance": 1,
    "fecha_validacion": datetime.now().isoformat(),
    "total_registros": 505650,
    "integridad_referencial": "Correcta",
    "viajes_analizados": 100000,
    "fechas_incorrectas": 0,
    "llegadas_pendientes": 0,
    "viajes_sobrecargados": 0,
    "viajes_sin_datos_capacidad": 0,
    "entregas_analizadas": 400000,
    "seguimientos_vacios": 0,
    "seguimientos_duplicados": 0,
    "resultado_general": "Validaciones aprobadas"
}

with open("reporte_calidad_avance1.json", "w", encoding="utf-8") as archivo:
    json.dump(reporte, archivo, indent=4, ensure_ascii=False)

print("✅ Reporte de calidad generado correctamente")

✅ Reporte de calidad generado correctamente


In [2]:
import os

print("Archivo encontrado:", os.path.exists("reporte_calidad_avance1.json"))
print("Ubicación:", os.path.abspath("reporte_calidad_avance1.json"))

Archivo encontrado: True
Ubicación: /home/rocio/Proyectos/FleetLogix/reporte_calidad_avance1.json


In [5]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT
            tc.table_name,
            kcu.column_name,
            ccu.table_name AS tabla_referenciada,
            ccu.column_name AS columna_referenciada
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.table_schema = kcu.table_schema
        JOIN information_schema.constraint_column_usage ccu
            ON tc.constraint_name = ccu.constraint_name
            AND tc.table_schema = ccu.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = 'public'
        ORDER BY tc.table_name;
    """)
    relaciones = cursor.fetchall()

for relacion in relaciones:
    print(relacion)

('deliveries', 'trip_id', 'trips', 'trip_id')
('maintenance', 'vehicle_id', 'vehicles', 'vehicle_id')
('trips', 'vehicle_id', 'vehicles', 'vehicle_id')
('trips', 'driver_id', 'drivers', 'driver_id')
('trips', 'route_id', 'routes', 'route_id')


In [11]:
import psycopg2
from getpass import getpass

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="fleetlogix",
    user="fleetlogix_user",
    password=getpass("Contraseña PostgreSQL: ")
)

print("✅ Conexión a PostgreSQL exitosa")

Contraseña PostgreSQL:  ········


✅ Conexión a PostgreSQL exitosa


## Diagrama entidad-relación (ER) — FleetLogix

El modelo relacional está compuesto por 6 tablas y 5 relaciones mediante claves foráneas. El diagrama fue generado a partir de PostgreSQL utilizando DBeaver.

## Diagrama Entidad-Relación (ER) — FleetLogix

El siguiente diagrama representa las seis tablas de la base de datos PostgreSQL y sus relaciones mediante claves primarias (PK) y foráneas (FK).

![Diagrama ER de FleetLogix](FleetLogix_Diagrama_ER_Avance1.png)

### Relaciones y cardinalidades del modelo

El modelo relacional de FleetLogix contiene cinco relaciones de tipo uno a muchos (1:N):

1. **vehicles → trips (1:N):** un vehículo puede realizar múltiples viajes, pero cada viaje pertenece a un vehículo.

2. **drivers → trips (1:N):** un conductor puede realizar múltiples viajes, pero cada viaje tiene un conductor asignado.

3. **routes → trips (1:N):** una ruta puede utilizarse en múltiples viajes, pero cada viaje corresponde a una ruta.

4. **trips → deliveries (1:N):** un viaje puede contener múltiples entregas, pero cada entrega pertenece a un viaje.

5. **vehicles → maintenance (1:N):** un vehículo puede tener múltiples registros de mantenimiento, pero cada mantenimiento corresponde a un vehículo.

Estas relaciones están implementadas mediante claves foráneas (FK) en PostgreSQL y permiten mantener la integridad referencial de los datos.

### Reglas de negocio — FleetLogix

El sistema contempla las siguientes reglas de negocio:

1. Cada viaje debe estar asociado a un vehículo, un conductor y una ruta registrados.

2. El peso total transportado en un viaje no debe superar la capacidad máxima del vehículo asignado.

3. La fecha de llegada de un viaje no puede ser anterior a su fecha de salida.

4. Cada entrega debe estar asociada a un viaje existente y contar con un número de seguimiento único.

5. Cada mantenimiento debe estar asociado a un vehículo registrado.

### Validaciones realizadas

Se verificaron 505.650 registros distribuidos en seis tablas.

Los controles ejecutados no detectaron:
- Relaciones huérfanas.
- Viajes con fechas inconsistentes.
- Viajes con sobrecarga.
- Números de seguimiento vacíos o duplicados.

Los resultados se documentaron en el archivo `reporte_calidad_avance1.json`.